# Установка пакетов и загрузка данных для ноутбука

Для работы нам понадобится установить пакет `waymax` https://github.com/waymo-research/waymax


Waymax -- симулятор для исследования автономного вождения, базирующийся на `Waymo Open Motion Dataset` https://github.com/waymo-research/waymo-open-dataset

Для обучения сетки будем использовать
`Pytorch Lightning` https://lightning.ai/docs/pytorch/

Для запуска на GPU нужно сменить среду выполнения на Графический процессор

Устанавливаем пакеты:

In [ ]:
!pip install --upgrade pip
!pip install git+https://github.com/waymo-research/waymax.git@main#egg=waymo-waymax
!pip install pytorch-lightning==2.5.0

Семпл данных для семинара лежит тут https://disk.yandex.ru/d/IoFBUM-OHDKh4w

**Обратите внимание, архив отличается от архива с семинара наличием тестов и размером датасета**


In [ ]:
!wget -O data.tar.gz "$(curl -s "https://cloud-api.yandex.net/v1/disk/public/resources/download?public_key=https://disk.yandex.ru/d/IoFBUM-OHDKh4w" | jq -r .href)"
!mkdir ysda-prediction
!tar -xf data.tar.gz -C ysda-prediction

In [ ]:
SEMINAR_PATH = 'ysda-prediction'

Копируем себе в локальное файловое пространство папки с полезными функциями

In [ ]:
import os
import shutil

if not os.path.exists('lib'):
    shutil.copytree(os.path.join(SEMINAR_PATH, 'lib'), 'lib')
else:
    print('"lib" folder already exists. If you want to rewrite lib by original folder, remove local "lib" manually')

Импортируем библиотеки

In [ ]:
%%capture

import os
from copy import deepcopy
from typing import Any, Optional, Callable, Sequence
from collections import defaultdict
import dataclasses

import shapely

import mediapy
import numpy as np
import matplotlib.pyplot as plt
from matplotlib import patches
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

import jax
from jax import numpy as jnp
from jax import random

import pytorch_lightning as pl
from pytorch_lightning.loggers import TensorBoardLogger

from waymax.config import DatasetConfig
from waymax import visualization
from waymax import datatypes
from waymax import config as _config
from waymax import dataloader
from waymax import dynamics
from waymax import metrics
from waymax import env as _env
from waymax import agents
from waymax.agents import SimAgentActor, WaymaxActorOutput

import chex

from tqdm import tqdm


from lib.data_utils import WaymaxDataset, scenario_to_features_gt
from lib.tests.tests import test_metrics


# Use CUDA for hw
device = 'cuda'

С помощью следующей функции генерируем конфиг для подгрузки датасета

In [ ]:
def get_data_config(split_name, seminar_path=SEMINAR_PATH):
    split_path = os.path.join(SEMINAR_PATH, 'data', split_name)

    obj_count = int(os.listdir(split_path)[0].rsplit('-')[-1])
    return DatasetConfig(
        path=os.path.join(split_path, f'{split_name}_tfexample.tfrecord@{obj_count}'),
        max_num_objects=24,
        batch_dims=[8],
        repeat=1,
        shuffle_buffer_size=64,
        deterministic=False,
        num_shards=1
    )

Создаем датасеты для разных сплитов

In [ ]:
train_dataset = WaymaxDataset(get_data_config('training'))
val_dataset = WaymaxDataset(get_data_config('validation'))

In [ ]:
FUTURE_STEPS = 30  # Predict 3 seconds,  10Hz frequency

In [ ]:
scenario = next(iter(train_dataset))
features, gt = scenario_to_features_gt(scenario)

### Вcпомогательные функции отрисовки предсказаний модели

Код отрисовки сцены

In [ ]:
def plot_states(states, use_log_traj=False, batch_idx=0):
    imgs = []
    for state in states:
        imgs.append(visualization.plot_simulator_state(
            state, use_log_traj=use_log_traj, batch_idx=batch_idx))
    mediapy.show_video(imgs, fps=10)


def extract_best_mode_from_pred_component(pred, best_mode):
    return pred[np.arange(best_mode.shape[0]), best_mode]

def extract_first_timestep_from_pred(pred):
    return pred[..., 0]


# generate open loop prediction with single model run
def generate_open_loop(model, scenario, history_size=11, agent_idx=None):
    states = []

    # some tensors for slicing
    batch_range = jnp.arange(scenario.object_metadata.is_sdc.shape[0])
    control_mask = scenario.object_metadata.is_sdc
    if agent_idx:
        control_mask = jnp.zeros_like(control_mask)
        control_mask = control_mask.at[batch_range, agent_idx].set(True)
        print(f'Control agent with agent_idx = {agent_idx}')
    else:
        print(f'Control ego agent')
    control_indices = control_mask.argmax(-1)

    features, gt = scenario_to_features_gt(
        scenario,
        agent_to_predict_mask=control_mask,
        device=device
    )

    model.to(device)

    # run model
    pred = model(features)
    state = scenario

    # get best mode
    best_mode = pred['logits'].argmax(-1)

    # get best by logit predictions and convert to numpy
    new_x = extract_best_mode_from_pred_component(pred['trajectory']['x'], best_mode).detach().cpu().numpy()
    new_y = extract_best_mode_from_pred_component(pred['trajectory']['y'], best_mode).detach().cpu().numpy()
    new_vel_x = extract_best_mode_from_pred_component(pred['trajectory']['vel_x'], best_mode).detach().cpu().numpy()
    new_vel_y = extract_best_mode_from_pred_component(pred['trajectory']['vel_y'], best_mode).detach().cpu().numpy()
    new_yaw = extract_best_mode_from_pred_component(pred['trajectory']['yaw'], best_mode).detach().cpu().numpy()


    # make 10 steps to make step.timestep equal to first inference timestep
    state = datatypes.update_state_by_log(state, num_steps=10)
    assert (state.timestep == 10).all()


    for _ in range(FUTURE_STEPS):
        # copy log trajectories to sim trajectories for all objects
        state = datatypes.update_state_by_log(state, num_steps=1)

        # rewrite ego sim_trajectory by predictions
        state.sim_trajectory.x = state.sim_trajectory.x.at[batch_range, control_indices, history_size:history_size+FUTURE_STEPS].set(new_x)
        state.sim_trajectory.y = state.sim_trajectory.y.at[batch_range, control_indices, history_size:history_size+FUTURE_STEPS].set(new_y)
        state.sim_trajectory.yaw = state.sim_trajectory.yaw.at[batch_range, control_indices, history_size:history_size+FUTURE_STEPS].set(new_yaw)
        state.sim_trajectory.vel_x = state.sim_trajectory.vel_x.at[batch_range, control_indices, history_size:history_size+FUTURE_STEPS].set(new_vel_x)
        state.sim_trajectory.vel_y = state.sim_trajectory.vel_y.at[batch_range, control_indices, history_size:history_size+FUTURE_STEPS].set(new_vel_y)

        state.sim_trajectory.valid = state.sim_trajectory.valid.at[batch_range, control_indices, history_size:history_size+FUTURE_STEPS].set(True)

        states.append(state)

    return states

Код отрисовки мод

In [ ]:
def plot_modes(scenario, planner_model, batch_idx):
    features, gt = scenario_to_features_gt(scenario, map_points=2048)
    pred = planner_model(features)

    plt.figure(figsize=(10, 10))

    # draw roadgraph
    rg_points = features['roadgraph_points']
    where_valid = rg_points['valid'][batch_idx]

    rg_x = rg_points['x'][batch_idx][where_valid]
    rg_y = rg_points['y'][batch_idx][where_valid]
    types = rg_points['types'][batch_idx][where_valid]

    plt.scatter(
        x=rg_x,
        y=rg_y,
        s=0.1,
        c=types
    )

    # draw agents
    for agent_idx in range(features['log_trajectory']['x'].shape[-1]):
        x = features['log_trajectory']['x'][batch_idx, agent_idx, -1].detach().cpu().numpy()
        y = features['log_trajectory']['y'][batch_idx, agent_idx, -1].detach().cpu().numpy()
        yaw = features['log_trajectory']['yaw'][batch_idx, agent_idx, -1].detach().cpu().numpy()
        length = features['log_trajectory']['length'][batch_idx, agent_idx, -1].detach().cpu().numpy()
        width = features['log_trajectory']['width'][batch_idx, agent_idx, -1].detach().cpu().numpy()
        valid = features['log_trajectory']['valid'][batch_idx, agent_idx, -1].detach().cpu().numpy()

        is_ego = features['object_metadata']['is_sdc'][batch_idx, agent_idx]

        if not valid:
            continue

        x_corner_offset = length / 2
        y_corner_offset = width / 2

        r = patches.Rectangle(
                xy=(x - x_corner_offset, y - y_corner_offset),
                width=length,
                height=width,
                color='cyan' if is_ego else 'black',
                angle=yaw * 180 / np.pi,
                rotation_point='xy'
            )

        plt.gca().add_patch(r)

    # # draw prediction
    for mode_idx in range(pred['trajectory']['x'].shape[1]):

        plt.scatter(
            pred['trajectory']['x'][batch_idx, mode_idx].detach().cpu().numpy(),
            pred['trajectory']['y'][batch_idx, mode_idx].detach().cpu().numpy(),
            s=5,
            label=f'Mode {mode_idx}'
        )

    scale = 100
    plt.xlim(rg_x.min(), rg_x.max())
    plt.ylim(rg_y.min(), rg_y.max())

    plt.legend()
    plt.show()

# Задания

## 1. Метрики [1 балл]

Для начала реализуем функцию подсчета метрик, нас интересуют `top1_ade` и `min_ade` для предсказываемого агента `gt['agent_to_predict_mask']`. Не забудьте учесть маску видимости агента в гт, усредняем только те таймстемпы, в которые агент наблюдался.

In [ ]:
def prediction_metrics(pred, gt):
    """
    Compute loss for first mode

    Args:
      pred: prediction
      gt: target

    Returns:
      loss: dict with loss
    """
    agent_to_predict_mask = gt['agent_to_predict_mask']
    # YOUR CODE HERE

    return {
        'min_ade': ,
        'top1_ade': ,
    }

Тест на 1 балл

In [ ]:
test_metrics(prediction_metrics)

## 2. Planning модель [9 баллов]

Реализуйте модель для планирования эго-траектории.

**Обязательно переводите в локальную систему координат фичи, которые приходят в сетку, по аналогии с задачей 1 из дз по Prediction. Предсказывайте траектории в локальной системе координат и только потом делайте трансформ в оригинальную систему координат**

Модель должна предсказывать 6 мод по 3 секунды(и вероятности мод) и иметь формат выхода аналогичный с SimpleModel

Баллы за качество:
- min_ade=0.6, top1_ade=1.5 [9 баллов]
- min_ade=0.7, top1_ade=1.75 [7 баллов]
- min_ade=0.8, top1_ade=2.0 [5 балла]

Нужно приложить обучающие кривые (можно скрины) и вывод метрик на валидации. Кроме того, обязательно нарисуйте лучшие предсказания вашей модели(возьмите самые интересные и удачные на ваш взгляд сцены). За отсутствие метрик / кривых / визуализаций будем снимать баллы.

Сохраните чекпоинт для следующей ДЗ, вам предстоит зафайнтюнить его в CL обучении

Пример `SimpleModel`(заведомо плохая модель из-за отсутствия нормализации фичей в локальную систему координат) по аналогии с семинарской моделью, используйте ее формат входа и выхода

In [ ]:
class SimpleModel(nn.Module):
    def __init__(
            self,
            n_modes=6,
            future_steps=FUTURE_STEPS,
            hidden_dim=128,
            history_size=11,
    ):
        """
        Simple model

        """
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(history_size * 7 + 1, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, future_steps * n_modes * 6 + n_modes)
        )
        self.future_steps = future_steps
        self.n_modes = n_modes

    def __call__(self, features):
        bs, n_a, n_t = features['log_trajectory']['x'].shape

        # Make agent history features
        agents_history = torch.stack(
            [
                features['log_trajectory']['x'],
                features['log_trajectory']['y'],
                features['log_trajectory']['vel_x'],
                features['log_trajectory']['vel_y'],
                features['log_trajectory']['length'],
                features['log_trajectory']['width'],
                features['log_trajectory']['valid'],
            ],
            axis=-1
        )

        # Take interesting agent features
        agent_history = agents_history[features['agent_to_predict_mask']]
        agent_history = agent_history.reshape(bs, -1)

        # Add agent_type feature to features
        agent_type = features['object_metadata']['object_types'][features['agent_to_predict_mask']]
        agent_features = torch.cat(
            [agent_history, agent_type[..., None]], dim=-1)


        # Make prediction
        res = self.layers(agent_features)

        logits = res[..., -self.n_modes:]
        traj = res[..., :-self.n_modes].reshape(logits.shape[0], self.n_modes, self.future_steps, 6)

        res = {
            'trajectory': {
                'x': traj[..., 0],  # [bs, n_modes, future_steps]
                'y': traj[..., 1],  # [bs, n_modes, future_steps]
                'yaw': torch.atan2(traj[..., 2], traj[..., 3]),  # [bs, n_modes, future_steps]
                'vel_x': traj[..., 4],  # [bs, n_modes, future_steps]
                'vel_y': traj[..., 5]  # [bs, n_modes, future_steps]
            },
            'logits': logits  # [bs, n_modes]
        }

        return res

Класс для обучения



In [ ]:
class PredictionModule(pl.LightningModule):
    """
    Module with methods for model training
    """
    def __init__(
        self,
        model,
        loss_components_function,
        metrics_function,
        training_shift_random_generator=lambda : 0,
        training_agent_sampler=lambda scenario: scenario.object_metadata.is_sdc,
        validation_agent_sampler=lambda scenario: scenario.object_metadata.is_sdc,
        map_points=256,
        lr=1e-3
    ):
        """
        Args:
          model: model to train
          loss_components_function:
            function that takes (pred, gt) and return dict of losses
          metrics_function:
            function that takes (pred, gt) and return dict of metrics
          training_shift_random_generator:
            function that return shift for features/gt split
            it allow us to augment dataset and get more diverse scenarios
          training_agent_sampler: function that return mask with agent of interes during training
          validation_agent_sampler: function that return mask with agent of interes during validation
        """
        super().__init__()
        self.model = model
        self.loss_components_function = loss_components_function
        self.metrics_function = metrics_function
        self.training_shift_random_generator = training_shift_random_generator
        self.training_agent_sampler = training_agent_sampler
        self.validation_agent_sampler = validation_agent_sampler

        self.map_points = map_points
        self.lr = lr

    # Method to log loss
    def log_loss(self, split_name, loss, loss_components, batch_size):
        """
        Log loss to tensorboard

        Args:
          split_name: train/val
          loss: step loss
          loss_components: dict of step loss components
          batch_size: batch_size
        """
        self.log(
            f'{split_name}/epoch/loss/loss',
            loss.detach().cpu().item(),
            batch_size=batch_size,
            on_epoch=True,
            on_step=False
        )
        for k, v in loss_components.items():
            self.log(
                f'{split_name}/epoch/loss/{k}',
                v.detach().cpu().item(),
                batch_size=batch_size,
                on_epoch=True,
                on_step=False
            )

    # Method to log metrics
    def log_metrics(self, split_name, metrics, batch_size):
        """
        Log metrics to tensorboard

        Args:
          split_name: train/val
          metrics: dict of step metrics
          batch_size: batch_size
        """
        for k, v in metrics.items():
            self.log(
                f'{split_name}/epoch/{k}',
                v.detach().cpu().item(),
                batch_size=batch_size,
                on_epoch=True,
                on_step=False
            )

    # Model step
    def step(self, scenario, agent_to_predict_mask, shift=0):
        """
        Make model step

        Args:
          scenario: train/val
          agent_to_predict_mask: mask with agent to predict
          shift: shift for features

        Returns:
          (
            pred,            # model output
            gt,              # target
            loss,            # loss value
            loss_components  # loss components dict
          )
        """
        # split scenario to features and gt
        features, gt = scenario_to_features_gt(
            scenario,
            map_points=self.map_points,
            features_first_timestamp=shift,
            agent_to_predict_mask=agent_to_predict_mask,
            device=self.device)
        # apply model
        pred = self.model(features)
        # compute losses
        loss_components = self.loss_components_function(pred, gt)
        loss = sum(loss_components[key] for key in loss_components.keys())
        return pred, gt, loss, loss_components

    # Validation step
    @torch.no_grad
    def validation_step(self, scenario, batch_idx):
        """
        Make validation step

        Args:
          scenario: train/val
          batch_idx: number of step

        Returns:
          loss
        """
        # sample agent to predict
        agent_to_predict_mask = self.validation_agent_sampler(scenario)

        pred, gt, loss, loss_components = self.step(
            scenario,
            agent_to_predict_mask=agent_to_predict_mask)
        self.log_loss(
            'val', loss, loss_components,
            batch_size=scenario['log_trajectory']['x'].shape[0],
        )
        self.log_metrics(
            'val', self.metrics_function(pred, gt),
            batch_size=scenario['log_trajectory']['x'].shape[0],)

        return loss

    # Training step
    def training_step(self, scenario, batch_idx):
        """
        Make training step

        Args:
          scenario: train/val
          batch_idx: number of step

        Returns:
          loss
        """
        # sample agent to predict
        agent_to_predict_mask = self.training_agent_sampler(scenario)

        pred, gt, loss, loss_components = self.step(
            scenario,
            agent_to_predict_mask=agent_to_predict_mask,
            shift=self.training_shift_random_generator())
        self.log_loss(
            'train', loss, loss_components,
            batch_size=scenario['log_trajectory']['x'].shape[0],
        )
        return loss

    # Make optimizer
    def configure_optimizers(self):
        """
        Create optimizer

        Returns:
          optimizer
        """
        optimizer = torch.optim.Adam(self.parameters(), lr=self.lr)
        return optimizer

Реализуйте лосс для предсказания агента, выделяемого маской `gt['agent_to_predict_mask']`. Не забудьте, что лосс должен включать в себя компоненты не только для позиций и логитов, но и для скорости с углом. Не забудьте учесть маску видимости агента в гт

In [ ]:
def planner_loss(pred, gt):
    """
    Compute loss for first mode

    Args:
      pred: prediction
      gt: target

    Returns:
      loss: dict with losses
    """
    agent_to_predict_mask = gt['agent_to_predict_mask']

    # YOUR CODE HERE

    return {
        # YOUR CODE HERE
    }


In [ ]:
%reload_ext tensorboard
%tensorboard --logdir=tb_logs/

Инициализируем модуль и тренера.

**Обратите внимание, что по дефолту используется 256 токенов карты, этого может оказаться мало, при необходимости увеличивайте**

In [ ]:
planner_model = SimpleModel(future_steps=FUTURE_STEPS)

planner = PredictionModule(
    model=planner_model,
    loss_components_function=planner_loss,
    metrics_function=prediction_metrics,
    training_shift_random_generator=lambda : np.random.randint(0, 80 - FUTURE_STEPS + 1),
    training_agent_sampler=lambda x: x.object_metadata.is_sdc,
    validation_agent_sampler=lambda x: x.object_metadata.is_sdc,
    map_points=256,
    lr=5e-4
)
planner = planner.to(device)
logger = TensorBoardLogger("tb_logs", name="planner_model")

trainer = pl.Trainer(logger=logger, log_every_n_steps=1, gradient_clip_val=0.5)

Запускаем обучение

In [ ]:
trainer.fit(model=planner, train_dataloaders=train_dataset, val_dataloaders=val_dataset)

Считаем метрики на валидации

In [ ]:
trainer.validate(model=planner, dataloaders=val_dataset)

Рисуем предсказания

In [ ]:
scenario = next(iter(val_dataset))

In [ ]:
# predict ego
states = generate_open_loop(planner.model, scenario)

In [ ]:
plot_states(states, batch_idx=0)

In [ ]:
plot_states(states, batch_idx=1)

In [ ]:
plot_modes(scenario, planner_model, batch_idx=0)

In [ ]:
plot_modes(scenario, planner_model, batch_idx=0)